In [18]:
!pip install -q transformers sentencepiece torch

In [19]:
from transformers import pipeline
import re

# 1. 意圖分類：判斷客服信件類型
intent_classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

# 2. 中文情緒分析
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="uer/roberta-base-finetuned-jd-binary-chinese"
)

# 3. 中文 NER 實體抽取
ner_extractor = pipeline(
    "token-classification",
    model="ckiplab/bert-base-chinese-ner",
    aggregation_strategy="simple"
)

# 4. 多語摘要
summarizer = pipeline(
    "summarization",
    model="csebuetnlp/mT5_multilingual_XLSum"
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: uer/roberta-base-finetuned-jd-binary-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ckiplab/bert-base-chinese-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [20]:
message = """
您好，我在 2026/05/29 購買了 Orbit X1 降噪耳機，訂單編號 OD-583920。
商品昨天到貨後左耳完全沒有聲音，我已經嘗試重置還是無法使用。
這是送給主管的生日禮物，現在真的很尷尬。
請協助我辦理退貨退款，並告知多久會退回信用卡。
"""

result = customer_ai_pipeline(message)

原始客服信件

您好，我在 2026/05/29 購買了 Orbit X1 降噪耳機，訂單編號 OD-583920。
商品昨天到貨後左耳完全沒有聲音，我已經嘗試重置還是無法使用。
這是送給主管的生日禮物，現在真的很尷尬。
請協助我辦理退貨退款，並告知多久會退回信用卡。

Step 1 意圖分類
預測類別：退貨退款
信心分數：0.8080

Step 2 情緒分析
情緒：negative (stars 1, 2 and 3)
信心分數：0.9826

Step 3 關鍵實體抽取
{'entity': 'DATE', 'word': '2026/05/', 'score': 1.0}
{'entity': 'DATE', 'word': '29', 'score': 1.0}
{'entity': 'CARDINAL', 'word': '5839', 'score': 0.8937}
{'entity': 'CARDINAL', 'word': '##20', 'score': 0.7595}
{'entity': 'DATE', 'word': '昨', 'score': 1.0}
{'entity': 'DATE', 'word': '天', 'score': 1.0}

Step 3.5 訂單資訊抽取
{'dates': ['2026/05/29'], 'orders': ['OD-583920'], 'product': 'Orbit X1 降噪耳機'}

Step 4 信件摘要
您好，我在 2026/05/29 購買了 Orbit X1 降噪耳機，訂單編號 OD-583920。 商品昨天到貨後左耳完全沒有聲音，我已經嘗試重置還是無法使用。 這是送給主管的生日禮物，現在真的很尷...

Step 5 回覆草稿
您好，感謝您的來信。

很抱歉讓您有不佳的使用體驗。根據您的描述，您的問題主要屬於「退貨退款」，目前系統判斷情緒偏向「negative (stars 1, 2 and 3)」。

我們已注意到您的問題重點：
您好，我在 2026/05/29 購買了 Orbit X1 降噪耳機，訂單編號 OD-583920。 商品昨天到貨後左耳完全沒有聲音，我已經嘗試重置還是無法使用。 這是送給主管的生日禮物，現在真的很尷...

相關資訊包含：
2026/05/、29、

In [21]:
result

{'intent': {'label': '退貨退款',
  'score': 0.807956337928772,
  'full_result': {'sequence': '\n您好，我在 2026/05/29 購買了 Orbit X1 降噪耳機，訂單編號 OD-583920。\n商品昨天到貨後左耳完全沒有聲音，我已經嘗試重置還是無法使用。\n這是送給主管的生日禮物，現在真的很尷尬。\n請協助我辦理退貨退款，並告知多久會退回信用卡。\n',
   'labels': ['退貨退款', '帳務付款', '產品技術問題', '物流配送', '一般客服詢問'],
   'scores': [0.807956337928772,
    0.12577767670154572,
    0.03642510622739792,
    0.016405129805207253,
    0.013435741886496544]}},
 'sentiment': {'label': 'negative (stars 1, 2 and 3)',
  'score': 0.9825925827026367,
  'raw_result': {'label': 'negative (stars 1, 2 and 3)',
   'score': 0.9825925827026367}},
 'entities': [{'entity': 'DATE', 'word': '2026/05/', 'score': 1.0},
  {'entity': 'DATE', 'word': '29', 'score': 1.0},
  {'entity': 'CARDINAL', 'word': '5839', 'score': 0.8937},
  {'entity': 'CARDINAL', 'word': '##20', 'score': 0.7595},
  {'entity': 'DATE', 'word': '昨', 'score': 1.0},
  {'entity': 'DATE', 'word': '天', 'score': 1.0}],
 'order_info': {'dates': ['2026/05/29'],
  'orders': ['OD-583920'

In [22]:
def generate_simple_report(result):
    report = f"""
# AI 客服信件分析與回覆草稿系統

## 一、專題目標

本專題使用 Hugging Face Transformers Pipeline 建立一套客服信件分析流程，
將多種 NLP 任務串聯成自動化工作流，協助客服人員快速判斷客戶問題類型、
分析情緒、抽取關鍵資訊、摘要信件內容，並產生初步回覆草稿。

## 二、使用的 Pipeline

1. zero-shot-classification：判斷客服信件意圖
2. sentiment-analysis：分析客戶情緒
3. token-classification / NER：抽取信件中的關鍵實體
4. summarization：摘要客服信件重點

## 三、系統流程

原始客服信件
→ 意圖分類
→ 情緒分析
→ NER 實體抽取
→ 訂單資訊抽取
→ 信件摘要
→ 回覆草稿生成

## 四、分析結果

### 1. 意圖分類

預測類別：{result["intent"]["label"]}
信心分數：{result["intent"]["score"]:.4f}

### 2. 情緒分析

情緒：{result["sentiment"]["label"]}
信心分數：{result["sentiment"]["score"]:.4f}

### 3. NER 實體抽取

{result["entities"]}

### 4. 訂單資訊抽取

{result["order_info"]}

### 5. 信件摘要

{result["summary"]}

### 6. 客服回覆草稿

{result["reply"]}

## 五、系統價值

此系統可協助客服人員快速掌握客戶問題重點，
降低人工閱讀成本，並提供一致且具禮貌的初步回覆草稿。

## 六、限制與改進方向

1. 中文 NER 對訂單編號與產品型號辨識不一定穩定，因此加入規則式抽取補強。
2. 摘要模型在短文本上可能效果有限，仍需人工確認。
3. 後續可加入 FAQ 知識庫，讓系統根據公司政策產生更精準的回覆。
4. 未來可使用 Streamlit 建立互動式網頁介面。
"""

    return report


project_report = generate_simple_report(result)
print(project_report)


# AI 客服信件分析與回覆草稿系統

## 一、專題目標

本專題使用 Hugging Face Transformers Pipeline 建立一套客服信件分析流程，
將多種 NLP 任務串聯成自動化工作流，協助客服人員快速判斷客戶問題類型、
分析情緒、抽取關鍵資訊、摘要信件內容，並產生初步回覆草稿。

## 二、使用的 Pipeline

1. zero-shot-classification：判斷客服信件意圖
2. sentiment-analysis：分析客戶情緒
3. token-classification / NER：抽取信件中的關鍵實體
4. summarization：摘要客服信件重點

## 三、系統流程

原始客服信件
→ 意圖分類
→ 情緒分析
→ NER 實體抽取
→ 訂單資訊抽取
→ 信件摘要
→ 回覆草稿生成

## 四、分析結果

### 1. 意圖分類

預測類別：退貨退款  
信心分數：0.8080

### 2. 情緒分析

情緒：negative (stars 1, 2 and 3)  
信心分數：0.9826

### 3. NER 實體抽取

[{'entity': 'DATE', 'word': '2026/05/', 'score': 1.0}, {'entity': 'DATE', 'word': '29', 'score': 1.0}, {'entity': 'CARDINAL', 'word': '5839', 'score': 0.8937}, {'entity': 'CARDINAL', 'word': '##20', 'score': 0.7595}, {'entity': 'DATE', 'word': '昨', 'score': 1.0}, {'entity': 'DATE', 'word': '天', 'score': 1.0}]

### 4. 訂單資訊抽取

{'dates': ['2026/05/29'], 'orders': ['OD-583920'], 'product': 'Orbit X1 降噪耳機'}

### 5. 信件摘要

您好，我在 2026/05/29 購買了 Orbit X1 降噪耳機，訂單編號 OD-583920。 商品昨天到貨後左耳完全沒有聲音，我已經嘗試重置還是無